# Тест максимального контекста — Qwen3.5-4B

Цель: определить реальное потребление GPU-памяти при обучении на примерах с разной длиной контекста.

In [1]:
from pathlib import Path
import json
import torch
import os

# os.environ["HF_HOME"] = "/shared/hf_cache"

ROOT = Path("/workdir")
MODEL_ID = "Qwen/Qwen3.5-4B"
SEED_DIR = ROOT / "llm_fc_hypernym_prediction/output/dataset/seed_42"
MAX_LEN = 22000
NUM_TOP_SAMPLES = 20

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("SEED_DIR:", SEED_DIR)
print("TARGET_MODEL:", MODEL_ID)
print("MAX_LEN:", MAX_LEN)
print("NUM_TOP_SAMPLES:", NUM_TOP_SAMPLES)

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


torch: 2.6.0+cu124
cuda: True NVIDIA A100-SXM4-80GB
SEED_DIR: /workdir/llm_fc_hypernym_prediction/output/dataset/seed_42
TARGET_MODEL: Qwen/Qwen3.5-4B
MAX_LEN: 22000
NUM_TOP_SAMPLES: 20


# Загрузка и первичный анализ данных

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


train_records_raw = load_jsonl(SEED_DIR / "train.jsonl")
eval_records_raw = load_jsonl(SEED_DIR / "eval.jsonl")
print(f"train: {len(train_records_raw)}, eval: {len(eval_records_raw)}")

train: 2504, eval: 355


# Туллзы и токенизатор

In [3]:
import copy
import sys

PROJ_ROOT = ROOT / "llm_fc_hypernym_prediction"
sys.path.insert(0, str(PROJ_ROOT))
from utils.tools import hyponym_only

TOOLS = hyponym_only


def normalize_messages_for_template(messages):
    msgs = copy.deepcopy(messages)
    for msg in msgs:
        if msg.get("role") != "assistant":
            continue
        for tc in msg.get("tool_calls") or []:
            fn = tc.get("function", tc)
            args = fn.get("arguments")
            if isinstance(args, str):
                fn["arguments"] = json.loads(args)
    return msgs

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)


def count_tokens(messages: list[dict]) -> int:
    normalized = normalize_messages_for_template(messages)
    result = tokenizer.apply_chat_template(
        normalized,
        tools=TOOLS,
        tokenize=True,
        return_dict=True,
        enable_thinking=False,
    )
    return len(result["input_ids"])

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Подсчёт длин всех примеров

In [5]:
def annotate_with_length(records: list[dict]) -> list[dict]:
    for r in records:
        r["_n_tokens"] = count_tokens(r["messages"])
    return records


train_records_raw = annotate_with_length(train_records_raw)
eval_records_raw = annotate_with_length(eval_records_raw)

train_sorted = sorted(train_records_raw, key=lambda r: r["_n_tokens"], reverse=True)
eval_sorted = sorted(eval_records_raw, key=lambda r: r["_n_tokens"], reverse=True)

train_within_limit = [r for r in train_sorted if r["_n_tokens"] <= MAX_LEN]
eval_within_limit = [r for r in eval_sorted if r["_n_tokens"] <= MAX_LEN]

print(f"Всего train: {len(train_sorted)}, из них <= {MAX_LEN}: {len(train_within_limit)}")
print(f"Всего eval: {len(eval_sorted)}, из них <= {MAX_LEN}: {len(eval_within_limit)}")

top_train = train_within_limit[:NUM_TOP_SAMPLES]
top_eval = eval_within_limit[:NUM_TOP_SAMPLES]

print(f"\n=== Топ-{len(top_train)} train (<= {MAX_LEN}) ===")
for i, r in enumerate(top_train):
    print(f"  {i+1:2d}. {r['_n_tokens']:>6d} tokens")

print(f"\n=== Топ-{len(top_eval)} eval (<= {MAX_LEN}) ===")
for i, r in enumerate(top_eval):
    print(f"  {i+1:2d}. {r['_n_tokens']:>6d} tokens")

Всего train: 2504, из них <= 22000: 2504
Всего eval: 355, из них <= 22000: 355

=== Топ-20 train (<= 22000) ===
   1.  20176 tokens
   2.  20068 tokens
   3.  20040 tokens
   4.  19922 tokens
   5.  19888 tokens
   6.  19886 tokens
   7.  19885 tokens
   8.  19883 tokens
   9.  19844 tokens
  10.  19831 tokens
  11.  19814 tokens
  12.  19813 tokens
  13.  19798 tokens
  14.  19752 tokens
  15.  19750 tokens
  16.  19674 tokens
  17.  19654 tokens
  18.  19620 tokens
  19.  19447 tokens
  20.  19327 tokens

=== Топ-20 eval (<= 22000) ===
   1.  19538 tokens
   2.  19364 tokens
   3.  19363 tokens
   4.  18864 tokens
   5.  18344 tokens
   6.  18210 tokens
   7.  18160 tokens
   8.  18100 tokens
   9.  17913 tokens
  10.  17733 tokens
  11.  17653 tokens
  12.  17583 tokens
  13.  17210 tokens
  14.  17197 tokens
  15.  17181 tokens
  16.  17068 tokens
  17.  16358 tokens
  18.  16356 tokens
  19.  14988 tokens
  20.  14391 tokens


# Подготовка датасетов из топ-N примеров

In [6]:
from datasets import Dataset

records_for_train = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in top_train
]
records_for_eval = [
    {
        "messages": normalize_messages_for_template(r["messages"]),
        "tools": TOOLS,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    for r in top_eval
]

ds_train = Dataset.from_list(records_for_train)
ds_eval = Dataset.from_list(records_for_eval)

print(f"ds_train: {len(ds_train)}, ds_eval: {len(ds_eval)}")
print(f"Train token lengths: {[r['_n_tokens'] for r in top_train]}")
print(f"Eval token lengths: {[r['_n_tokens'] for r in top_eval]}")

ds_train: 20, ds_eval: 20
Train token lengths: [20176, 20068, 20040, 19922, 19888, 19886, 19885, 19883, 19844, 19831, 19814, 19813, 19798, 19752, 19750, 19674, 19654, 19620, 19447, 19327]
Eval token lengths: [19538, 19364, 19363, 18864, 18344, 18210, 18160, 18100, 17913, 17733, 17653, 17583, 17210, 17197, 17181, 17068, 16358, 16356, 14988, 14391]


# Загрузка модели Qwen3.5-4B

In [7]:
import gc
from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

base_mem = torch.cuda.memory_allocated() / 1e9
print(f"Модель загружена. Занято памяти: {base_mem:.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 33288.13it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:01<00:00, 422.77it/s]


Модель загружена. Занято памяти: 8.41 GB


# LoRA конфигурация

In [8]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

lora_mem = torch.cuda.memory_allocated() / 1e9
print(f"После LoRA: {lora_mem:.2f} GB (+{lora_mem - base_mem:.2f} GB)")

trainable params: 1,572,864 || all params: 4,207,324,160 || trainable%: 0.0374
После LoRA: 8.42 GB (+0.01 GB)


# Пошаговое тестирование памяти

In [9]:
from trl import SFTConfig, SFTTrainer

gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = ROOT / "output" / "ctx_mem_test"

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=4e-4,
    bf16=True,
    logging_steps=10,
    save_steps=50,
    max_length=MAX_LEN,
    assistant_only_loss=True,
    gradient_checkpointing=True,
    save_total_limit=3,
    loss_type="chunked_nll",
    report_to="none",

    # eval_strategy="steps",
    # eval_steps=50,
    # eval_on_start=True,
    # per_device_eval_batch_size=1,

    # load_best_model_at_end=True,
    # metric_for_best_model="loss",
)


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    processing_class=tokenizer,
)

print(f"Датасет: {len(trainer.train_dataset)} примеров")

[2026-07-05 21:40:31,786] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
Building labels for train dataset: 100%|██████████| 20/20 [00:00<00:00, 39.52 examples/s]


Датасет: 20 примеров


In [10]:
import torch
import pandas as pd
from transformers import TrainerCallback

class GpuMemProfilerCallback(TrainerCallback):
    """Собирает peak VRAM на каждом микро-батче во время train()."""

    def __init__(self, seq_lens: list[int] | None = None):
        self.results: list[dict] = []
        self._micro_idx = 0
        # Длины из уже токенизированного датасета (точнее, чем _n_tokens)
        self.seq_lens = seq_lens

    def on_train_begin(self, args, state, control, **kwargs):
        torch.cuda.reset_peak_memory_stats()
        self._micro_idx = 0
        print(f"GPU mem profiling started. Total VRAM: "
              f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    def _snapshot(self, state, tag: str):
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated() / 1e9
        current = torch.cuda.memory_allocated() / 1e9

        seq_len = None
        if self.seq_lens and self._micro_idx < len(self.seq_lens):
            seq_len = self.seq_lens[self._micro_idx]

        row = {
            "micro_idx": self._micro_idx,
            "global_step": state.global_step,
            "tag": tag,
            "seq_len": seq_len,
            "peak_gb": round(peak, 3),
            "current_gb": round(current, 3),
        }
        self.results.append(row)
        print(
            f"micro={self._micro_idx:>2d} | global_step={state.global_step:>2d} | "
            f"seq={seq_len or '?':>6} | peak={peak:.2f} GB | current={current:.2f} GB"
        )
        self._micro_idx += 1
        torch.cuda.reset_peak_memory_stats()

    def on_step_begin(self, args, state, control, **kwargs):
        # Сброс счётчика пика перед группой микро-батчей
        torch.cuda.reset_peak_memory_stats()

    def on_substep_end(self, args, state, control, **kwargs):
        # Микро-батчи 1..GAS-1: backward уже прошёл, optimizer ещё нет
        self._snapshot(state, tag="substep")

    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        # Последний микро-батч в группе: максимум памяти перед optimizer.step()
        self._snapshot(state, tag="pre_optim")

    def on_train_end(self, args, state, control, **kwargs):
        if not self.results:
            print("Нет данных.")
            return
        max_row = max(self.results, key=lambda r: r["peak_gb"])
        print(f"\nМакс. peak: {max_row['peak_gb']:.2f} GB "
              f"(micro={max_row['micro_idx']}, seq={max_row['seq_len']})")


# Длины из токенизированного датасета trainer'а
tokenized_lens = [len(row["input_ids"]) for row in trainer.train_dataset]

mem_profiler = GpuMemProfilerCallback(seq_lens=tokenized_lens)
trainer.add_callback(mem_profiler)

In [11]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


GPU mem profiling started. Total VRAM: 85.0 GB
micro= 0 | global_step= 0 | seq= 20176 | peak=42.45 GB | current=8.44 GB
micro= 1 | global_step= 0 | seq= 20068 | peak=41.81 GB | current=8.44 GB
micro= 2 | global_step= 0 | seq= 20040 | peak=42.49 GB | current=8.44 GB
micro= 3 | global_step= 0 | seq= 19922 | peak=42.70 GB | current=8.44 GB


Step,Training Loss


micro= 4 | global_step= 1 | seq= 19888 | peak=42.17 GB | current=8.46 GB
micro= 5 | global_step= 1 | seq= 19886 | peak=42.26 GB | current=8.46 GB
micro= 6 | global_step= 1 | seq= 19885 | peak=42.17 GB | current=8.46 GB
micro= 7 | global_step= 1 | seq= 19883 | peak=42.05 GB | current=8.46 GB
micro= 8 | global_step= 2 | seq= 19844 | peak=42.17 GB | current=8.46 GB
micro= 9 | global_step= 2 | seq= 19831 | peak=42.07 GB | current=8.46 GB

Макс. peak: 42.70 GB (micro=3, seq=19922)


TrainOutput(global_step=3, training_loss=0.12415168682734172, metrics={'train_runtime': 1453.6119, 'train_samples_per_second': 0.014, 'train_steps_per_second': 0.002, 'total_flos': 8536026502901760.0, 'train_loss': 0.12415168682734172, 'entropy': 0.07895242907106877, 'num_tokens': 396072.0, 'mean_token_accuracy': 0.9694513320922852, 'epoch': 1.0})

In [12]:
df = pd.DataFrame(mem_profiler.results)
display(df)

print(f"Макс. peak: {df['peak_gb'].max():.2f} GB")
if df['seq_len'].notna().any():
    worst = df.loc[df['peak_gb'].idxmax()]
    print(f"Худший пример: seq_len={worst['seq_len']}, peak={worst['peak_gb']:.2f} GB")

,micro_idx,global_step,tag,seq_len,peak_gb,current_gb
0,0,0,substep,20176,42.454,8.445
1,1,0,substep,20068,41.812,8.445
2,2,0,substep,20040,42.485,8.445
3,3,0,pre_optim,19922,42.697,8.445
4,4,1,substep,19888,42.172,8.458
5,5,1,substep,19886,42.263,8.458
6,6,1,substep,19885,42.174,8.458
7,7,1,pre_optim,19883,42.047,8.458
8,8,2,substep,19844,42.172,8.456
9,9,2,pre_optim,19831,42.070,8.456


Макс. peak: 42.70 GB
Худший пример: seq_len=19922, peak=42.70 GB


In [13]:
# import torch
# torch.cuda.reset_peak_memory_stats()
# for i, batch in enumerate(trainer.get_train_dataloader()):
#     trainer.training_step(model, batch)
#     seq_len = batch["input_ids"].shape[1]
#     peak = torch.cuda.max_memory_allocated() / 1e9
#     print(f"step {i}: seq={seq_len}, peak={peak:.2f} GB")
#     torch.cuda.reset_peak_memory_stats()

In [14]:
# gc.collect()
# torch.cuda.empty_cache()

# baseline_mem = torch.cuda.memory_allocated() / 1e9
# print(f"Baseline: {baseline_mem:.2f} GB\n")

# results = []

# for i, batch in enumerate(trainer.get_train_dataloader()):
#     seq_len = batch["input_ids"].shape[1]

#     torch.cuda.reset_peak_memory_stats()
#     try:
#         loss = trainer.training_step(model, batch)
#     except torch.cuda.OutOfMemoryError:
#         peak = torch.cuda.max_memory_allocated() / 1e9
#         results.append({
#             "step": i,
#             "seq_len": seq_len,
#             "peak_gb": round(peak, 3),
#             "current_gb": "OOM",
#         })
#         print(f"step {i:3d}: seq={seq_len:>6d}, peak={peak:.3f} GB, *** OOM ***")
#         del batch
#         torch.cuda.empty_cache()
#         gc.collect()
#         continue

#     peak = torch.cuda.max_memory_allocated() / 1e9
#     current = torch.cuda.memory_allocated() / 1e9

#     results.append({
#         "step": i,
#         "seq_len": seq_len,
#         "peak_gb": round(peak, 3),
#         "current_gb": round(current, 3),
#     })
#     print(f"step {i:3d}: seq={seq_len:>6d}, peak={peak:.3f} GB, current={current:.3f} GB")

#     del loss, batch
#     gc.collect()

# Сводная таблица

In [15]:
# print(f"{'step':>5} {'seq_len':>8} {'peak GB':>10} {'current GB':>11}")
# print("-" * 38)
# for r in results:
#     cur = r['current_gb'] if isinstance(r['current_gb'], str) else f"{r['current_gb']:.3f}"
#     print(f"{r['step']:>5} {r['seq_len']:>8} {r['peak_gb']:>10.3f} {cur:>11}")

# if results:
#     max_peak = max(r["peak_gb"] for r in results)
#     max_seq_ok = max(r["seq_len"] for r in results if r["current_gb"] != "OOM")
#     oom_count = sum(1 for r in results if r["current_gb"] == "OOM")
#     print(f"\nМакс. пиковая память: {max_peak:.3f} GB")
#     print(f"Макс. seq_len без OOM: {max_seq_ok} токенов")
#     if oom_count:
#         print(f"OOM на {oom_count} из {len(results)} примерах")

# Оценка потолка контекста

In [16]:
# total_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
# max_peak = max(r["peak_gb"] for r in results) if results else 0
# remaining = total_mem_gb - max_peak

# ok_results = [r for r in results if r["current_gb"] != "OOM"]

# print(f"Всего VRAM:       {total_mem_gb:.1f} GB")
# print(f"Пик при обучении: {max_peak:.3f} GB")
# print(f"Свободно:         {remaining:.3f} GB")
# print()

# if ok_results:
#     avg_tokens_per_gb = sum(r["seq_len"] for r in ok_results) / len(ok_results) / max_peak
#     max_ok_seq = max(r["seq_len"] for r in ok_results)
#     print(f"Успешных примеров: {len(ok_results)}/{len(results)}")
#     print(f"Среднее: ~{avg_tokens_per_gb:.0f} токенов на GB VRAM")
#     est_max_ctx = int(remaining / max_peak * max_ok_seq) if max_peak > 0 else 0
#     print(f"Макс. seq_len без OOM: {max_ok_seq}")
#     print(f"Грубая оценка макс. контекста: ~{est_max_ctx} токенов")
# elif results:
#     print(f"Все примеры вызвали OOM — макс. контекста < {min(r['seq_len'] for r in results)} токенов")